# T8 - Calibration

This tutorial drafts how we would set up a simple calibration workflow:

- define which data we want to fit the model to 
- set up a simulation object
- define which parameters and which parameter values or ranges we want to test
- set up the calibration oject
- run the calibration

In [ ]:
import starsim as ss
import sciris as sc
import typhoidsim as ty

We usually have some data we'd like to fit the model to. The data below is simulated data using specific values for `init_prev` (0.01) and `p_cpg` (0.35). The simplest way to pass data for a calibration is in the form of a dataframe with column names matching quantities tracked by the `Sim`. For instance, `n_alive`, or `typhoid.n_chronic` -- the cumulative number of chroi


In [ ]:
def make_data():
    target_data = [
    ['year', 'n_alive', 'typhoid.prevalence', 'typhoid.n_infected',  'typhoid.n_chronic'],
    [  1982,  10_000,        0.603843, 	6033.0, 	82.0],
    [  1983,   9991.0,       0.844966, 	8426.0, 	135.0],
    [  1984,   9972.0,       0.939415, 	9350.0, 	202.0],
    [  1985,   9953.0,       0.975244, 	9691.0, 	262.0],
    [  1986,   9937.0,       0.990617, 	9819.0, 	317.0],
    [  1987,   9912.0,       0.997472, 	9863.0, 	379.0],
    [  1988,   9888.0,       0.998784, 	9855.0, 	442.0],
    [  1990,   9867.0,       0.999391, 	9843.0, 	500.0],
    [  1991,   9849.0 ,      0.999593, 	9823.0, 	567.0],
    ]
    df = sc.dataframe(target_data[1:], columns=target_data[0])
    return df

We first make a function that will create a Sim object with the ingredients we believe explain our data, and over a span of time that matches that of our data. Inside this function you will see that we have wrapped the basic setup seen in many of the other tutorials.

In [ ]:
def make_sim():
    pars = dict(
    start    = 1982,  # Starting year
    n_years  = 9.0,   # Duration of the simulation in years
    dt       = 1.0/365.0,     # Timestep of 1 day, expressed in years
    verbose  = 0,             # Do not print details of the run
)
    typhoid = ty.Typhoid()
    ppl = ss.People(10_000)

    sim = ss.Sim(
        pars=pars,
        people=ppl,
        diseases = typhoid,
    )

    return sim

Then, we specifiy which parameters need to be calibrated, and what are the ranges to explore. 
The calibration parameters has a hierarchical/nested structure similar to the Sim class. 

In [ ]:
def get_calib_pars():
    calib_pars = dict(
        diseases = dict(
            typhoid = dict(
                init_prev = [0.00, 0.01, 0.05],  # [low value, step, max value]
                p_cpg=[0.05, 0.05, 0.4],
            ),
        ),
    )
    return calib_pars


Now we set up the put all the elements together using Starsim's `Calibration` class that uses [optuna](https://optuna.org/)

In [ ]:
def run_calibration():
    sc.heading('Testing calibration')

    # The parameters or parameters of each ingredient need to exist in the sim defined in make_sim()

    # Make the sim and data
    sim = make_sim()
    data = make_data()
    calib_pars = get_calib_pars()

    # Make the calibration
    calib = ss.Calibration(
        calib_pars = calib_pars,
        db_name="typhoidsim_tutorial",
        sim = sim,
        data = data,
        total_trials = 50,
        n_workers = 4, # the underlying library runs in parallel
        die = True
    )

    # Perform the calibration
    sc.printcyan('\nPeforming calibration...')
    calib.calibrate(confirm_fit=False)

    # Confirm
    sc.printcyan('\nConfirming fit...')
    calib.confirm_fit()
    print(f'Fit with original pars: {calib.before_fit:n}')
    print(f'Fit with best-fit pars: {calib.after_fit:n}')
    if calib.after_fit <= calib.before_fit:
        print('✅ Calibration improved fit')
    else:
        print('❌ Calibration did not improve fit, but this sometimes happens stochastically and is not necessarily an error')
    
    return sim, calib

In [ ]:
run_calibration()